# 09 — Benchmark LLM Open-Weight vs Pipeline V9\n## POC R&D : évaluation des modèles génératifs pour la détection de désinformation\n\n**Objectif** : Comparer le pipeline ThumaCheck V9 (classification spécialisée, ~1.5 ms/texte) à des LLM open-weight en zero-shot sur le gold test set (465 textes Bluesky).\n\n**Hypothèse** : Le pipeline V9 domine en latence et explicabilité ; le LLM apporte une justification en langage naturel potentiellement utile en analyse batch offline (roadmap V11).\n\n**Modèles testés** :\n| Modèle | Type | Accès |\n|--------|------|-------|\n| **ThumaCheck V9** | Pipeline cascade (LogReg V5 + TF-IDF + features linguistiques) | Local |\n| **Mistral** | LLM souverain européen — candidat roadmap V12 | API |\n| **Tout modèle OpenAI-compatible** | GLM, Kimi K3, Qwen, etc. | API configurable |\n\n**Métriques** : F1 macro, Precision, Recall, latence médiane, coût estimé / 1 000 textes\n\n---\n*Notebook R&D — ThumaCheck / Niamato Consulting — Juillet 2026*

## 1. Setup & Configuration

In [ ]:
import sys, os, time, json, warnings\nimport pandas as pd\nimport numpy as np\nfrom pathlib import Path\n\nwarnings.filterwarnings(\"ignore\")\n\n# --- Chemins projet ---\nPROJECT_ROOT = Path(os.getcwd()).parent\nSRC_DIR = PROJECT_ROOT / \"src\"\nDATA_DIR = PROJECT_ROOT / \"data\"\nMODEL_DIR = PROJECT_ROOT / \"models\"\n\nsys.path.insert(0, str(SRC_DIR))\n\n# --- Sklearn metrics ---\nfrom sklearn.metrics import (\n    accuracy_score, f1_score, precision_score, recall_score,\n    roc_auc_score, confusion_matrix, classification_report\n)\n\n# --- Visualisation ---\nimport matplotlib.pyplot as plt\nimport matplotlib\nmatplotlib.rcParams.update({\"figure.dpi\": 120, \"font.size\": 11})\n\nprint(f\"Project root : {PROJECT_ROOT}\")\nprint(f\"Python       : {sys.version.split()[0]}\")

In [ ]:
# --- Configuration API (à renseigner avant exécution) ---\n# Option 1 : variables d'environnement\n# export MISTRAL_API_KEY=\"sk-...\"\n# export OPENAI_COMPAT_API_KEY=\"sk-...\"\n# export OPENAI_COMPAT_BASE_URL=\"https://api.example.com/v1\"\n# export OPENAI_COMPAT_MODEL=\"glm-5.2\"\n\n# Option 2 : renseigner ici directement\nCONFIG = {\n    \"mistral\": {\n        \"api_key\": os.environ.get(\"MISTRAL_API_KEY\", \"\"),\n        \"model\": os.environ.get(\"MISTRAL_MODEL\", \"mistral-small-latest\"),\n        \"base_url\": \"https://api.mistral.ai/v1\",\n        \"cost_per_1k_input\": 0.001,   # $/1k tokens (ajuster selon modèle)\n        \"cost_per_1k_output\": 0.003,\n    },\n    \"openai_compat\": {\n        \"api_key\": os.environ.get(\"OPENAI_COMPAT_API_KEY\", \"\"),\n        \"model\": os.environ.get(\"OPENAI_COMPAT_MODEL\", \"gpt-4o-mini\"),\n        \"base_url\": os.environ.get(\"OPENAI_COMPAT_BASE_URL\", \"https://api.openai.com/v1\"),\n        \"cost_per_1k_input\": 0.00015,\n        \"cost_per_1k_output\": 0.0006,\n    },\n}\n\nprint(\"Mistral API key :\", \"✅ configurée\" if CONFIG[\"mistral\"][\"api_key\"] else \"❌ manquante\")\nprint(\"OpenAI-compat key :\", \"✅ configurée\" if CONFIG[\"openai_compat\"][\"api_key\"] else \"❌ manquante\")

## 2. Chargement du Gold Test Set (465 textes Bluesky)

In [ ]:
gold = pd.read_csv(DATA_DIR / \"gold_test_set.csv\")\n\nprint(f\"Gold test set : {len(gold)} textes\")\nprint(f\"Colonnes      : {list(gold.columns)}\")\nprint(f\"\\nRépartition labels (ia_prediction) :\")\nprint(gold[\"ia_prediction\"].value_counts().rename({0: \"FIABLE (0)\", 1: \"SUSPECT (1)\"}))\nprint(f\"\\nRépartition langues :\")\nprint(gold[\"langue\"].value_counts())\nprint(f\"\\nLongueur texte (mots) :\")\nprint(gold[\"nb_mots\"].describe().round(1))\n\ngold.head(3)

## 3. Benchmark — Pipeline ThumaCheck V9 (baseline)\n\nChargement du modèle local V5 + prédiction sur les 465 textes avec mesure de latence.

In [ ]:
from pipeline.expert_detector import ExpertFakeNewsDetector\n\ndetector = ExpertFakeNewsDetector(model_dir=str(MODEL_DIR))\ndetector.load(suffix=\"expert_v5\")\nprint(\"✅ Pipeline V9 chargé\")

In [ ]:
# --- Prédictions V9 avec mesure de latence ---\ntexts = gold[\"text\"]\ny_true = gold[\"ia_prediction\"].values\n\n# Warm-up (1er appel peut être plus lent)\n_ = detector.predict(texts.head(5))\n\n# Benchmark\nlatencies_v9 = []\nfor i in range(3):  # 3 runs pour stabiliser\n    t0 = time.perf_counter()\n    results_v9 = detector.predict(texts)\n    t1 = time.perf_counter()\n    latencies_v9.append((t1 - t0) / len(texts) * 1000)  # ms/texte\n\ny_pred_v9 = results_v9[\"prediction_label\"].values\ny_score_v9 = results_v9[\"ai_score_credibility\"].values\n\nprint(f\"Latence V9 : {np.median(latencies_v9):.2f} ms/texte (médiane sur 3 runs)\")\nprint(f\"Throughput  : {1000 / np.median(latencies_v9):.0f} textes/sec\")\nprint(f\"\\n{classification_report(y_true, y_pred_v9, target_names=['FIABLE', 'SUSPECT'])}\")

## 4. Benchmark — LLM Zero-Shot\n\n### Prompt engineering\nUn prompt unique zero-shot est envoyé à chaque LLM. Le modèle doit répondre en JSON structuré avec un label (`fiable` / `suspect`) et un score de crédibilité entre 0 et 1.\n\n### Gestion des erreurs\n- Retry automatique (3 tentatives, backoff exponentiel)\n- Sauvegarde incrémentale toutes les 50 prédictions\n- Reprise possible depuis le dernier checkpoint

In [ ]:
import httpx, re\n\nSYSTEM_PROMPT = \"\"\"Tu es un expert en détection de désinformation sur les réseaux sociaux.\nAnalyse le texte suivant et détermine s'il est FIABLE ou SUSPECT (potentielle désinformation).\n\nRéponds UNIQUEMENT avec un objet JSON valide, sans texte autour :\n{\"label\": \"fiable\" ou \"suspect\", \"score\": float entre 0.0 et 1.0, \"justification\": \"explication courte\"}\n\nOù score = probabilité que le texte soit fiable (1.0 = très fiable, 0.0 = très suspect).\"\"\"\n\n\ndef call_llm_api(text: str, config: dict, timeout: float = 30.0) -> dict:\n    \"\"\"Appel générique compatible OpenAI API (Mistral, GLM, Kimi, etc.).\"\"\"\n    headers = {\n        \"Authorization\": f\"Bearer {config['api_key']}\",\n        \"Content-Type\": \"application/json\",\n    }\n    payload = {\n        \"model\": config[\"model\"],\n        \"messages\": [\n            {\"role\": \"system\", \"content\": SYSTEM_PROMPT},\n            {\"role\": \"user\", \"content\": text[:4000]},\n        ],\n        \"temperature\": 0.0,\n        \"max_tokens\": 200,\n    }\n    url = f\"{config['base_url'].rstrip('/')}/chat/completions\"\n\n    for attempt in range(3):\n        try:\n            resp = httpx.post(url, json=payload, headers=headers, timeout=timeout)\n            resp.raise_for_status()\n            content = resp.json()[\"choices\"][0][\"message\"][\"content\"]\n            usage = resp.json().get(\"usage\", {})\n            return {\"content\": content, \"usage\": usage, \"error\": None}\n        except Exception as e:\n            if attempt < 2:\n                time.sleep(2 ** attempt)\n            else:\n                return {\"content\": None, \"usage\": {}, \"error\": str(e)}\n\n\ndef parse_llm_response(raw: str) -> dict:\n    \"\"\"Parse la réponse JSON du LLM, avec fallback regex.\"\"\"\n    if not raw:\n        return {\"label\": None, \"score\": None, \"justification\": None}\n    try:\n        match = re.search(r'\\{[^}]+\\}', raw, re.DOTALL)\n        if match:\n            data = json.loads(match.group())\n            label = data.get(\"label\", \"\").lower().strip()\n            score = float(data.get(\"score\", 0.5))\n            return {\n                \"label\": 0 if label == \"fiable\" else (1 if label == \"suspect\" else None),\n                \"score\": np.clip(score, 0.0, 1.0),\n                \"justification\": data.get(\"justification\", \"\"),\n            }\n    except (json.JSONDecodeError, ValueError, TypeError):\n        pass\n    return {\"label\": None, \"score\": None, \"justification\": raw[:200]}\n\n\nprint(\"✅ Fonctions LLM prêtes\")

In [ ]:
def run_llm_benchmark(gold_df: pd.DataFrame, config: dict, model_name: str,\n                      checkpoint_path: Path = None) -> pd.DataFrame:\n    \"\"\"Lance le benchmark LLM sur tout le gold test set avec checkpointing.\"\"\"\n    n = len(gold_df)\n    results = []\n    total_input_tokens = 0\n    total_output_tokens = 0\n\n    # Reprise depuis checkpoint si existant\n    start_idx = 0\n    if checkpoint_path and checkpoint_path.exists():\n        cached = pd.read_csv(checkpoint_path)\n        results = cached.to_dict(\"records\")\n        start_idx = len(results)\n        print(f\"♻️  Reprise depuis checkpoint : {start_idx}/{n} déjà traités\")\n\n    for i in range(start_idx, n):\n        text = gold_df.iloc[i][\"text\"]\n        t0 = time.perf_counter()\n        raw = call_llm_api(text, config)\n        latency = (time.perf_counter() - t0) * 1000  # ms\n\n        parsed = parse_llm_response(raw[\"content\"])\n        usage = raw.get(\"usage\", {})\n        total_input_tokens += usage.get(\"prompt_tokens\", 0)\n        total_output_tokens += usage.get(\"completion_tokens\", 0)\n\n        results.append({\n            \"idx\": i,\n            \"label_pred\": parsed[\"label\"],\n            \"score\": parsed[\"score\"],\n            \"justification\": parsed[\"justification\"],\n            \"latency_ms\": latency,\n            \"error\": raw[\"error\"],\n        })\n\n        # Checkpoint toutes les 50 prédictions\n        if checkpoint_path and (i + 1) % 50 == 0:\n            pd.DataFrame(results).to_csv(checkpoint_path, index=False)\n\n        # Progress\n        if (i + 1) % 100 == 0 or i == n - 1:\n            errors = sum(1 for r in results if r[\"error\"])\n            print(f\"  [{model_name}] {i+1}/{n} — erreurs: {errors} — \"\n                  f\"latence moy: {np.mean([r['latency_ms'] for r in results]):.0f} ms\")\n\n    # Sauvegarde finale\n    if checkpoint_path:\n        pd.DataFrame(results).to_csv(checkpoint_path, index=False)\n\n    # Coût estimé\n    cost_input = total_input_tokens / 1000 * config.get(\"cost_per_1k_input\", 0)\n    cost_output = total_output_tokens / 1000 * config.get(\"cost_per_1k_output\", 0)\n    cost_total = cost_input + cost_output\n    cost_per_1k = cost_total / n * 1000 if n > 0 else 0\n\n    print(f\"\\n📊 {model_name} terminé — tokens: {total_input_tokens}+{total_output_tokens} \"\n          f\"— coût: ${cost_total:.4f} (${cost_per_1k:.4f}/1k textes)\")\n\n    return pd.DataFrame(results)\n\n\nprint(\"✅ Fonction benchmark LLM prête\")

### 4a. Lancer le benchmark Mistral\n> ⚠️ Nécessite une clé API Mistral valide. ~465 appels API (~5-15 min selon le modèle).

In [ ]:
# --- Benchmark Mistral ---\nif CONFIG[\"mistral\"][\"api_key\"]:\n    checkpoint_mistral = DATA_DIR / \"benchmark_mistral_checkpoint.csv\"\n    df_mistral = run_llm_benchmark(\n        gold, CONFIG[\"mistral\"],\n        model_name=f\"Mistral ({CONFIG['mistral']['model']})\",\n        checkpoint_path=checkpoint_mistral,\n    )\nelse:\n    print(\"⏭️  Mistral skippé (pas de clé API). Définir MISTRAL_API_KEY.\")\n    df_mistral = None

### 4b. Lancer le benchmark OpenAI-compatible\n> Configurable via `OPENAI_COMPAT_BASE_URL`, `OPENAI_COMPAT_MODEL`, `OPENAI_COMPAT_API_KEY`.

In [ ]:
# --- Benchmark OpenAI-compatible ---\nif CONFIG[\"openai_compat\"][\"api_key\"]:\n    checkpoint_compat = DATA_DIR / \"benchmark_openai_compat_checkpoint.csv\"\n    df_compat = run_llm_benchmark(\n        gold, CONFIG[\"openai_compat\"],\n        model_name=f\"OpenAI-compat ({CONFIG['openai_compat']['model']})\",\n        checkpoint_path=checkpoint_compat,\n    )\nelse:\n    print(\"⏭️  OpenAI-compat skippé (pas de clé API). Définir OPENAI_COMPAT_API_KEY.\")\n    df_compat = None

## 5. Métriques comparatives

In [ ]:
def compute_metrics(y_true, y_pred, y_score, latencies_ms, model_name, cost_per_1k=0.0):\n    \"\"\"Calcule les métriques pour un modèle donné.\"\"\"\n    valid = ~pd.isna(y_pred)\n    if valid.sum() == 0:\n        return None\n    yt, yp = y_true[valid], y_pred[valid].astype(int)\n    return {\n        \"Modèle\": model_name,\n        \"N valides\": int(valid.sum()),\n        \"N erreurs\": int((~valid).sum()),\n        \"Accuracy\": accuracy_score(yt, yp),\n        \"F1 macro\": f1_score(yt, yp, average=\"macro\"),\n        \"F1 SUSPECT\": f1_score(yt, yp, pos_label=1),\n        \"Precision\": precision_score(yt, yp, average=\"macro\"),\n        \"Recall\": recall_score(yt, yp, average=\"macro\"),\n        \"ROC-AUC\": roc_auc_score(yt, y_score[valid]) if y_score is not None else None,\n        \"Latence méd. (ms)\": np.median(latencies_ms),\n        \"Latence p95 (ms)\": np.percentile(latencies_ms, 95),\n        \"Coût/1k textes ($)\": cost_per_1k,\n    }\n\n\n# --- Pipeline V9 ---\nall_metrics = []\nall_metrics.append(compute_metrics(\n    y_true, y_pred_v9, y_score_v9,\n    latencies_ms=[np.median(latencies_v9)] * len(y_true),\n    model_name=\"ThumaCheck V9\",\n    cost_per_1k=0.0,\n))\n\n# --- Mistral ---\nif df_mistral is not None and len(df_mistral) > 0:\n    y_pred_m = df_mistral[\"label_pred\"].values\n    y_score_m = df_mistral[\"score\"].values\n    all_metrics.append(compute_metrics(\n        y_true, y_pred_m, y_score_m,\n        latencies_ms=df_mistral[\"latency_ms\"].values,\n        model_name=f\"Mistral ({CONFIG['mistral']['model']})\",\n    ))\n\n# --- OpenAI-compat ---\nif df_compat is not None and len(df_compat) > 0:\n    y_pred_c = df_compat[\"label_pred\"].values\n    y_score_c = df_compat[\"score\"].values\n    all_metrics.append(compute_metrics(\n        y_true, y_pred_c, y_score_c,\n        latencies_ms=df_compat[\"latency_ms\"].values,\n        model_name=f\"OpenAI-compat ({CONFIG['openai_compat']['model']})\",\n    ))\n\ndf_bench = pd.DataFrame([m for m in all_metrics if m is not None])\ndf_bench = df_bench.set_index(\"Modèle\")\n\n# Formatage\nfor col in [\"Accuracy\", \"F1 macro\", \"F1 SUSPECT\", \"Precision\", \"Recall\", \"ROC-AUC\"]:\n    if col in df_bench.columns:\n        df_bench[col] = df_bench[col].map(lambda x: f\"{x:.4f}\" if pd.notna(x) else \"—\")\nfor col in [\"Latence méd. (ms)\", \"Latence p95 (ms)\"]:\n    if col in df_bench.columns:\n        df_bench[col] = df_bench[col].map(lambda x: f\"{x:.1f}\")\n\ndf_bench

## 6. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))\ncolors = [\"#2196F3\", \"#FF9800\", \"#4CAF50\"]\n\n# --- Données pour les graphiques ---\nmodels_data = [(\"ThumaCheck V9\", y_pred_v9, y_score_v9, np.median(latencies_v9))]\nif df_mistral is not None and len(df_mistral) > 0:\n    valid_m = df_mistral[\"label_pred\"].notna()\n    models_data.append((\n        f\"Mistral\", df_mistral[\"label_pred\"].values,\n        df_mistral[\"score\"].values, df_mistral[\"latency_ms\"].median()\n    ))\nif df_compat is not None and len(df_compat) > 0:\n    models_data.append((\n        f\"OpenAI-compat\", df_compat[\"label_pred\"].values,\n        df_compat[\"score\"].values, df_compat[\"latency_ms\"].median()\n    ))\n\n# --- 6a. F1 macro par modèle ---\nax = axes[0]\nnames, f1s = [], []\nfor name, yp, ys, lat in models_data:\n    valid = ~pd.isna(yp)\n    if valid.sum() > 0:\n        names.append(name)\n        f1s.append(f1_score(y_true[valid], yp[valid].astype(int), average=\"macro\"))\nbars = ax.bar(names, f1s, color=colors[:len(names)], edgecolor=\"white\", linewidth=1.5)\nfor bar, val in zip(bars, f1s):\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,\n            f\"{val:.3f}\", ha=\"center\", fontweight=\"bold\")\nax.set_ylabel(\"F1 macro\")\nax.set_title(\"F1 macro — Comparaison\")\nax.set_ylim(0, 1.05)\nax.axhline(y=0.9, color=\"gray\", linestyle=\"--\", alpha=0.5, label=\"seuil 0.9\")\nax.legend()\n\n# --- 6b. Latence médiane (log scale) ---\nax = axes[1]\nlat_names, lats = [], []\nfor name, yp, ys, lat in models_data:\n    lat_names.append(name)\n    lats.append(lat)\nbars = ax.bar(lat_names, lats, color=colors[:len(lat_names)], edgecolor=\"white\", linewidth=1.5)\nfor bar, val in zip(bars, lats):\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,\n            f\"{val:.1f} ms\", ha=\"center\", fontweight=\"bold\", fontsize=9)\nax.set_ylabel(\"Latence médiane (ms)\")\nax.set_title(\"Latence — Comparaison\")\nax.set_yscale(\"log\")\n\n# --- 6c. Matrice de confusion V9 ---\nax = axes[2]\ncm = confusion_matrix(y_true, y_pred_v9)\nim = ax.imshow(cm, cmap=\"Blues\", aspect=\"auto\")\nfor i in range(2):\n    for j in range(2):\n        ax.text(j, i, str(cm[i, j]), ha=\"center\", va=\"center\",\n                fontsize=14, fontweight=\"bold\",\n                color=\"white\" if cm[i, j] > cm.max()/2 else \"black\")\nax.set_xticks([0, 1])\nax.set_yticks([0, 1])\nax.set_xticklabels([\"Prédit\\nFIABLE\", \"Prédit\\nSUSPECT\"])\nax.set_yticklabels([\"Réel\\nFIABLE\", \"Réel\\nSUSPECT\"])\nax.set_title(\"Matrice confusion — V9\")\n\nplt.tight_layout()\nplt.savefig(PROJECT_ROOT / \"reports\" / \"benchmark_llm_comparison.png\", dpi=150, bbox_inches=\"tight\")\nplt.show()\nprint(\"📊 Figure sauvegardée dans reports/benchmark_llm_comparison.png\")

### 6b. Analyse par langue (FR vs EN)

In [ ]:
# --- Métriques par langue ---\nlang_results = []\nfor lang in [\"fr\", \"en\"]:\n    mask = gold[\"langue\"] == lang\n    if mask.sum() == 0:\n        continue\n    yt_lang = y_true[mask]\n\n    # V9\n    f1_v9_lang = f1_score(yt_lang, y_pred_v9[mask], average=\"macro\")\n    lang_results.append({\"Langue\": lang.upper(), \"Modèle\": \"ThumaCheck V9\", \"F1 macro\": f1_v9_lang, \"N\": mask.sum()})\n\n    # Mistral\n    if df_mistral is not None:\n        yp_m = df_mistral[\"label_pred\"].values[mask]\n        valid = ~pd.isna(yp_m)\n        if valid.sum() > 0:\n            f1_m = f1_score(yt_lang[valid], yp_m[valid].astype(int), average=\"macro\")\n            lang_results.append({\"Langue\": lang.upper(), \"Modèle\": \"Mistral\", \"F1 macro\": f1_m, \"N\": valid.sum()})\n\n    # OpenAI-compat\n    if df_compat is not None:\n        yp_c = df_compat[\"label_pred\"].values[mask]\n        valid = ~pd.isna(yp_c)\n        if valid.sum() > 0:\n            f1_c = f1_score(yt_lang[valid], yp_c[valid].astype(int), average=\"macro\")\n            lang_results.append({\"Langue\": lang.upper(), \"Modèle\": \"OpenAI-compat\", \"F1 macro\": f1_c, \"N\": valid.sum()})\n\ndf_lang = pd.DataFrame(lang_results)\nif len(df_lang) > 0:\n    print(df_lang.pivot_table(index=\"Modèle\", columns=\"Langue\", values=\"F1 macro\").round(4).to_string())\nelse:\n    print(\"Pas assez de données pour l'analyse par langue.\")

### 6c. Analyse qualitative — Justifications LLM (échantillon)

In [ ]:
# --- Échantillon de justifications LLM vs V9 ---\ndef show_qualitative_sample(df_llm, llm_name, n=5):\n    \"\"\"Affiche un échantillon de prédictions divergentes entre V9 et LLM.\"\"\"\n    if df_llm is None:\n        print(f\"Pas de données pour {llm_name}\")\n        return\n\n    # Cas où V9 et LLM divergent\n    valid = df_llm[\"label_pred\"].notna()\n    divergent = valid & (df_llm[\"label_pred\"].values != y_pred_v9)\n    idxs = np.where(divergent)[0]\n\n    if len(idxs) == 0:\n        print(f\"Aucune divergence V9 vs {llm_name} !\")\n        return\n\n    sample = np.random.choice(idxs, size=min(n, len(idxs)), replace=False)\n    print(f\"\\n{'='*80}\")\n    print(f\"DIVERGENCES V9 vs {llm_name} ({len(idxs)} cas, échantillon de {len(sample)})\")\n    print(f\"{'='*80}\")\n\n    for idx in sample:\n        text = gold.iloc[idx][\"text\"][:150]\n        label_true = \"FIABLE\" if y_true[idx] == 0 else \"SUSPECT\"\n        label_v9 = \"FIABLE\" if y_pred_v9[idx] == 0 else \"SUSPECT\"\n        label_llm = \"FIABLE\" if df_llm.iloc[idx][\"label_pred\"] == 0 else \"SUSPECT\"\n        justif = df_llm.iloc[idx].get(\"justification\", \"—\")\n\n        correct_v9 = \"✅\" if y_pred_v9[idx] == y_true[idx] else \"❌\"\n        correct_llm = \"✅\" if df_llm.iloc[idx][\"label_pred\"] == y_true[idx] else \"❌\"\n\n        print(f\"\\n[#{idx}] Vérité: {label_true}\")\n        print(f\"  Texte : {text}...\")\n        print(f\"  V9    : {label_v9} {correct_v9} (score={y_score_v9[idx]:.3f})\")\n        print(f\"  {llm_name}: {label_llm} {correct_llm} (score={df_llm.iloc[idx]['score']:.3f})\")\n        print(f\"  Justif: {justif}\")\n\nnp.random.seed(42)\nshow_qualitative_sample(df_mistral, \"Mistral\")\nshow_qualitative_sample(df_compat, \"OpenAI-compat\")

## 7. Conclusion & Recommandations\n\n### Résultats attendus\n\n| Critère | Pipeline V9 | LLM zero-shot |\n|---------|------------|---------------|\n| **F1 macro** | ~0.91 (entraîné sur domaine) | ~0.70-0.85 (zero-shot) |\n| **Latence** | ~1.5 ms/texte | ~500-2000 ms/texte |\n| **Coût** | 0 $ (local) | ~0.05-0.50 $/1k textes |\n| **Explicabilité** | Coefficients + SHAP | Justification LN |\n| **Souveraineté** | 100% local | Dépend du provider |\n\n### Recommandations\n\n1. **Production temps réel** : Pipeline V9 reste le choix optimal (latence, coût, explicabilité)\n2. **Analyse batch offline (roadmap V11)** : Un LLM peut enrichir les analyses avec des justifications en langage naturel\n3. **Hybridation V12** : Utiliser le LLM en second avis sur les cas indécis (score V9 entre 0.35-0.55)\n4. **Mistral privilégié** : Modèle souverain européen, compatible RGPD, candidat roadmap\n\n### Limites de ce benchmark\n- Zero-shot uniquement (pas de few-shot ni fine-tuning)\n- Gold test set = 465 textes (taille limitée)\n- Coûts API variables selon le moment et le provider\n- Le prompt influence fortement les résultats LLM

In [ ]:
# --- Export résultats ---\noutput_dir = PROJECT_ROOT / \"reports\"\noutput_dir.mkdir(exist_ok=True)\n\ndf_bench.to_csv(output_dir / \"benchmark_llm_results.csv\")\nprint(f\"✅ Résultats exportés dans {output_dir / 'benchmark_llm_results.csv'}\")\n\n# Résumé final\nprint(f\"\\n{'='*60}\")\nprint(\"RÉSUMÉ BENCHMARK\")\nprint(f\"{'='*60}\")\nprint(f\"Gold test set      : {len(gold)} textes (FR+EN)\")\nprint(f\"Modèles testés     : {len(df_bench)}\")\nprint(f\"Meilleur F1 macro  : {df_bench.index[0]} → {df_bench.iloc[0]['F1 macro']}\")\nprint(f\"{'='*60}\")